In [11]:
import json
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..','..')))
from ai_tools.tools import LLMQuery
import gradio as gr
from ai_tools.tools import MODEL_DICT
import os
import pandas as pd
import io


  {
    "person_id": "d9f8a5b2-3c1e-4f6a-9b2e-7e4f1a2c9d3b",
    "first_name": "Anna",
    "last_name": "Müller",
    "gender": "female",
    "date_of_birth": "1988-03-14",
    "age": 37,
    "street_address": "Prenzlauer Allee 45, 3.OG",
    "postal_code": "10115",
    "city": "Berlin",
    "state": "Berlin",
    "citizenship": "DE",
    "tax_id": "95732184602",
    "health_insurance": "Techniker Krankenkasse",
    "employment_status": "employed_full_time",
    "gross_annual_income_eur": 72000
  ,
    "person_id": "a3c1b5f7-9d2e-4b6f-a1c8-2e7f3b9d4a5f",
    "first_name": "Maximilian",
    "last_name": "Schneider",
    "gender": "male",
    "date_of_birth": "1975-11-02",
    "age": 50,
    "street_address": "Sonnenstraße 12, Whg. 4",
    "postal_code": "80331",
    "city": "München",
    "state": "Bayern",
    "citizenship": "DE",
    "tax_id": "64159720358",
    "health_insurance": "AOK",
    "employment_status": "employed_full_time",
    "gross_annual_income_eur": 95000
  ,
    "per

In [6]:
SYSTEM_PROMPT = """### ROLE
You are the **Mock Data Generator Assistant**, a specialized AI designed to generate high-fidelity, domain-specific mock datasets based on vague or specific business problem statements.

### OBJECTIVE
Your goal is to interpret a user's business scenario (e.g., "HR attrition analysis," "Supply chain logistics," "E-commerce transaction logs") and generate a realistic dataset in **strict JSON format**.

### OPERATIONAL RULES

1.  **Analyze the Domain:** deeply understand the industry implied by the prompt. If the user asks for "hospital patients," include medical-specific fields like `diagnosis_code`, `admission_date`, `insurance_provider`, and `blood_type`.
2.  **Infer Attributes:** Do not wait for the user to list columns. You must infer the most valuable 10-15 attributes that a data scientist or developer would need to solve the specific business problem.
3.  **Enforce Realism:**
    * Do not use placeholder values like "User1", "Test Data", or "asdf".
    * Use realistic names, addresses, UUIDs, timestamps, and domain-specific jargon (e.g., SKU numbers, ICD-10 codes, Stock tickers).
    * Ensure logical consistency (e.g., `delivery_date` must be after `order_date`; `age` must match `date_of_birth`).
4.  **Volume:** Unless specified otherwise, generate **10 records** to provide a representative sample.

### FORMATTING CONSTRAINTS (CRITICAL)
* **Output:** You must return **ONLY** a valid JSON array of objects.
* **No Chatter:** Do not provide introductions, explanations, or "Here is your data" text. Start with `[` and end with `]`.
* **Syntax:** Ensure strict JSON compliance (double quotes for keys/strings, no trailing commas).
* **JSON Layout:** Always return an array of objects, never a single object.

### ERROR HANDLING
If the user request is gibberish or unrelated to data generation, return a JSON object with a single error field:
`[{"error": "Unable to generate data. Please provide a valid business scenario."}]`

### EXAMPLES

**User Input:** "I need to test a fraud detection system for credit cards."
**System Output:**
[
  {
    "transaction_id": "550e8400-e29b-41d4-a716-446655440000",
    "timestamp": "2023-10-27T14:30:00Z",
    "amount": 1250.00,
    "currency": "USD",
    "merchant": "Global Electronics",
    "merchant_category_code": 5732,
    "card_holder": "Jane Doe",
    "card_last_four": "4242",
    "device_ip": "192.168.1.1",
    "is_flagged": true,
    "risk_score": 0.89
  },
  {
    ...
  }
]"""

**FRONTIER MODELS**

In [7]:
mock_data_generator_frontier = LLMQuery(
    system_prompt= SYSTEM_PROMPT,
    json_format=True
)

In [ ]:

# Flatten model list for the dropdown
all_models = []
for models in MODEL_DICT.values():
    all_models.extend(list(models))
all_models.sort()

def generate_data(user_input, model_name):
    """
    Generates mock data based on user input and selected model.
    Returns a dataframe and a path to the CSV file.
    """
    if not user_input:
        return None, None

    try:
        # Query the LLM
        response = mock_data_generator_frontier.query(
            user_input, model=model_name, display_output=False, use_history=False
        )

        print(response)
        data = json.loads(response)
        # Ensure data is a list of objects for DataFrame
        if isinstance(data, dict):
            # If a single object is returned, wrap it in a list
            data = [data]

        df = pd.json_normalize(data)

        # Save to CSV for download
        # We save to a temporary file or a static one;
        # overwriting "generated_mock_data.csv" is fine for single user demo
        csv_filename = os.path.abspath(os.path.join(
            os.getcwd(), "mock_data.csv"
        ))
        df.to_csv(csv_filename, index=False)

        return df, csv_filename

    except Exception as e:
        # Return an error dataframe
        error_df = pd.DataFrame({"Error": [f"Failed to generate data: {str(e)}"]})
        return error_df, None


# Build the Gradio Interface
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎲 Frontier Models - Mock Data Generator")
    gr.Markdown(
        "Describe the data you need, select a model, and generate a downloadable dataset."
    )

    with gr.Row():
        with gr.Column(scale=1):
            model_selector = gr.Dropdown(
                choices=all_models,
                value="gemini-flash-latest",
                label="Select Model",
                info="Choose the LLM to generate your data.",
                interactive=True,
            )
        with gr.Column(scale=3):
            pass  # Spacer

    user_input = gr.Textbox(
        label="Data Description",
        placeholder="e.g., I need a dataset of 10 fake transactions for a fraud detection system by extracting json...",
        lines=2,
    )

    with gr.Row():
        generate_btn = gr.Button("🚀 Generate Data", variant="primary")
        stop_btn = gr.Button("🛑 Stop", variant="stop")

    output_df = gr.DataFrame(label="Generated Data", interactive=False, wrap=True)

    with gr.Row():
        download_btn = gr.DownloadButton("📥 Download CSV")

    # Event wiring
    # Click button
    click_event = generate_btn.click(
        fn=generate_data,
        inputs=[user_input, model_selector],
        outputs=[output_df, download_btn],
    )

    # Stop button wiring
    stop_btn.click(fn=None, inputs=None, outputs=None, cancels=[click_event])

    # Press Enter (submit)
    submit_event = user_input.submit(
        fn=generate_data,
        inputs=[user_input, model_selector],
        outputs=[output_df, download_btn],
    )

    # Allow stopping submit event as well
    stop_btn.click(fn=None, inputs=None, outputs=None, cancels=[submit_event])

demo.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


[
  {
    "inhabitant_id": "DE-89210-XJ",
    "first_name": "Maximilian",
    "last_name": "Müller",
    "gender": "Male",
    "date_of_birth": "1985-04-12",
    "place_of_birth": "Stuttgart",
    "address": "Hauptstraße 14",
    "postal_code": "10115",
    "city": "Berlin",
    "state": "Berlin",
    "phone_number": "+49 151 23456789",
    "email": "maximilian.mueller@web.de",
    "occupation": "Maschinenbauingenieur",
    "tax_id": "44-567-890-123",
    "iban": "DE45 1001 0010 1234 5678 90",
    "health_insurance": "Techniker Krankenkasse"
  },
  {
    "inhabitant_id": "DE-10492-AB",
    "first_name": "Sabine",
    "last_name": "Schmidt",
    "gender": "Female",
    "date_of_birth": "1992-09-23",
    "place_of_birth": "Dresden",
    "address": "Goethestraße 5",
    "postal_code": "80331",
    "city": "München",
    "state": "Bayern",
    "phone_number": "+49 170 98765432",
    "email": "s.schmidt92@gmx.net",
    "occupation": "Lehrerin",
    "tax_id": "89-123-456-789",
    "iban": "D